# M5.3 · Diffusion (DDPM) ve flow matching

Gürültü ekleyip geri almayı öğrenmek. İki adımlık bir ileri süreçte $x_t$'yi ve geri adımın ortalamasını elle hesaplıyor, saf Python'da tek boyutlu Gauss veri için kapalı formdaki "kâhin" gürültü tahmincisiyle DDPM örnekleyicisi yazıyor, NumPy'da tüm örnekleri paralel yürüten DDPM ve DDIM örnekleyicilerini kuruyor, PyTorch'ta bir gürültü tahmin ağı eğitip 8 Gauss'lu halkayı üretiyor ve en sonda flow matching ile aynı işi birkaç Euler adımında yapıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m5-3-diffusion.html](https://m-gonul.github.io/pytorch-lab/m5-3-diffusion.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m5_3_diffusion.py
CHECKS = {
 "b": "import math\nimport random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    R.test(\"alpha_bars\", lambda: close(need(ns, \"alpha_bars\")([0.1, 0.2, 0.5]), [0.9, 0.72, 0.36]))\n    ab = [0.9, 0.72]\n    R.test(\"q_sample (elle örnek)\", lambda: close(need(ns, \"q_sample\")(2.0, 1, 0.5, ab), 1.961632, atol=1e-6))\n\n    def step():\n        x2 = math.sqrt(0.72) * 2 + math.sqrt(0.28) * 0.5\n        close(need(ns, \"ddpm_step\")(x2, 1, 0.5, [0.1, 0.2], ab, 0.0), 1.981884, atol=1e-6, what=\"z = 0\")\n        close(ns[\"ddpm_step\"](x2, 1, 0.5, [0.1, 0.2], ab, 1.0), 1.981884 + math.sqrt(0.1 / 0.28 * 0.2), atol=1e-6, what=\"z = 1\")\n        close(ns[\"ddpm_step\"](1.0, 0, 0.3, [0.1, 0.2], ab, 5.0), (1 - 0.1 / math.sqrt(0.1) * 0.3) / math.sqrt(0.9), atol=1e-6, what=\"t = 0: gürültü yok\")\n    R.test(\"ddpm_step\", step)\n\n    def orc():\n        rng = random.Random(3)\n        abar = [0.3]\n        xs = [math.sqrt(0.3) * 1.5 + math.sqrt(0.7) * 0 for _ in range(1)]\n        pairs = []\n        for _ in range(40000):\n            x0 = rng.gauss(1.5, 0.8); e = rng.gauss(0, 1)\n            pairs.append((math.sqrt(0.3) * x0 + math.sqrt(0.7) * e, e))\n        xs = [p[0] for p in pairs]; es = [p[1] for p in pairs]\n        mx = sum(xs) / len(xs)\n        slope = sum((x - mx) * e for x, e in zip(xs, es)) / sum((x - mx) ** 2 for x in xs)\n        pred = need(ns, \"oracle_eps\")(mx + 1.0, 0, abar, 1.5, 0.8) - ns[\"oracle_eps\"](mx, 0, abar, 1.5, 0.8)\n        ensure(abs(pred - slope) < 0.02, f\"kâhinin eğimi {pred:.4f}, regresyon eğimi {slope:.4f}\")\n        ensure(abs(ns[\"oracle_eps\"](math.sqrt(0.3) * 1.5, 0, abar, 1.5, 0.8)) < 1e-12, \"ortalamada tahmin 0 olmalı\")\n    R.test(\"oracle_eps = E[ε | x_t] (regresyonla)\", orc)\n\n    def smp():\n        T = 100\n        betas = [1e-4 + (0.1 - 1e-4) * i / (T - 1) for i in range(T)]\n        abar = need(ns, \"alpha_bars\")(betas)\n        xs = need(ns, \"sample\")(lambda x, t: ns[\"oracle_eps\"](x, t, abar, 3.0, 0.5), betas, 3000, random.Random(0))\n        m = sum(xs) / len(xs); s = (sum((x - m) ** 2 for x in xs) / len(xs)) ** 0.5\n        ensure(abs(m - 3) < 0.06 and abs(s - 0.5) < 0.05, f\"ortalama {m:.3f}, std {s:.3f}; beklenen 3, 0.5\")\n        return f\"3000 örnek: ortalama {m:.3f}, std {s:.3f} (hedef 3, 0.5)\"\n    R.test(\"kâhinle örnekleme N(3, 0.5²) veriyor\", smp)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"q_sample döngüsüz\", lambda: max_loops(src, 0, func=\"q_sample\"))\n    R.test(\"ddpm_sample: tek döngü (zaman)\", lambda: max_loops(src, 1, func=\"ddpm_sample\", why=\"örnekler paralel\"))\n    R.test(\"ddim_sample: tek döngü (zaman)\", lambda: max_loops(src, 1, func=\"ddim_sample\", why=\"örnekler paralel\"))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    betas = np.linspace(1e-4, 0.05, 300)\n    abar = np.cumprod(1 - betas)\n    rng = np.random.default_rng(0)\n\n    def qs():\n        x0 = rng.normal(size=(6, 3)); t = np.array([0, 5, 299, 5, 50, 1]); e = rng.normal(size=(6, 3))\n        want = np.stack([np.sqrt(abar[ti]) * x + np.sqrt(1 - abar[ti]) * ei for x, ti, ei in zip(x0, t, e)])\n        close(need(ns, \"q_sample\")(x0, t, e, abar), want)\n    R.test(\"q_sample: örnek başına farklı t\", qs)\n    mu, s = np.array([2.0, -1.0]), np.array([0.5, 1.5])\n    oracle = lambda x, t: np.sqrt(1 - abar[t]) * (x - np.sqrt(abar[t]) * mu) / (abar[t] * s ** 2 + 1 - abar[t])\n\n    def dd():\n        x = need(ns, \"ddpm_sample\")(oracle, betas, 6000, 2, np.random.default_rng(1))\n        ensure(x.shape == (6000, 2), \"şekil (n, d)\")\n        ensure(np.abs(x.mean(0) - mu).max() < 0.06 and np.abs(x.std(0) - s).max() < 0.06, f\"ortalama {x.mean(0).round(3)}, std {x.std(0).round(3)}\")\n        return f\"ortalama {x.mean(0).round(3)}, std {x.std(0).round(3)} (hedef {mu}, {s})\"\n    R.test(\"DDPM (300 adım) kâhinle\", dd)\n\n    def di():\n        rows = []\n        for st in (10, 20, 50, 100):\n            x = need(ns, \"ddim_sample\")(oracle, betas, 6000, 2, np.random.default_rng(1), st)\n            rows.append([str(st), str(x.mean(0).round(3)), str(x.std(0).round(3))])\n        ensure(np.abs(x.mean(0) - mu).max() < 0.05 and np.abs(x.std(0) - s).max() < 0.05, f\"100 adım: ortalama {x.mean(0).round(3)}, std {x.std(0).round(3)}\")\n        y = ns[\"ddim_sample\"](oracle, betas, 6000, 2, np.random.default_rng(1), 100)\n        ensure(np.array_equal(x, y), \"aynı tohumla aynı sonuç (deterministik)\")\n        R.table(f\"DDIM, kâhin; hedef ortalama {mu}, std {s}\", [\"adım\", \"ortalama\", \"std\"], rows,\n                note=\"DDIM birinci mertebe bir ODE çözücüsü gibi davranır: az adımda varyans eksik kalır. DPM-Solver gibi yüksek mertebeli çözücüler bu yüzden var.\")\n    R.test(\"DDIM kâhinle (adım sayısı etkisi)\", di)\n\n    def det():\n        calls = []\n        def fn(x, t):\n            calls.append(t); return oracle(x, t)\n        g = np.random.default_rng(2)\n        need(ns, \"ddim_sample\")(fn, betas, 10, 2, g, 5)\n        want = np.linspace(299, 0, 5).round().astype(int).tolist()\n        ensure([int(c) for c in calls] == want, f\"zaman adımları {calls}, beklenen {want}\")\n    R.test(\"DDIM zaman adımları\", det)\n",
 "d": "import math\nimport torch\nfrom torch import nn\nfrom labcheck import close, need, ensure\n\n\ndef ring(n):\n    k = torch.randint(0, 8, (n,))\n    ang = k * 2 * math.pi / 8\n    return torch.stack([2 * ang.cos(), 2 * ang.sin()], 1) + 0.05 * torch.randn(n, 2)\n\n\nclass Oracle(nn.Module):\n    def __init__(self, abar, mu, s):\n        super().__init__(); self.abar, self.mu, self.s = abar, mu, s\n\n    def forward(self, x, t):\n        a = self.abar[t][:, None]\n        return (1 - a).sqrt() * (x - a.sqrt() * self.mu) / (a * self.s ** 2 + 1 - a)\n\n\ndef check(ns, R):\n    def sched():\n        b, a = need(ns, \"make_schedule\")(10, 0.1, 0.5)\n        close(b, torch.linspace(0.1, 0.5, 10)); close(a, torch.cumprod(1 - torch.linspace(0.1, 0.5, 10), 0))\n    R.test(\"make_schedule\", sched)\n\n    def emb():\n        e = need(ns, \"TimeEmbedding\", \"sınıf\")(8)(torch.tensor([0, 3]))\n        f = torch.exp(-math.log(1000) * torch.arange(4) / 4)\n        close(e, torch.cat([(torch.tensor([[0.0], [3.0]]) * f).sin(), (torch.tensor([[0.0], [3.0]]) * f).cos()], 1))\n    R.test(\"TimeEmbedding\", emb)\n\n    def loss():\n        torch.manual_seed(0)\n        m = need(ns, \"EpsNet\", \"sınıf\")()\n        _, abar = ns[\"make_schedule\"]()\n        x0 = torch.randn(6, 2); t = torch.tensor([0, 10, 99, 50, 3, 3]); e = torch.randn(6, 2)\n        a = abar[t][:, None]\n        want = ((m(a.sqrt() * x0 + (1 - a).sqrt() * e, t) - e) ** 2).mean()\n        close(need(ns, \"ddpm_loss\")(m, x0, abar, t=t, eps=e), want)\n    R.test(\"ddpm_loss formülü (t, eps verilince)\", loss)\n\n    def oracle():\n        betas, abar = need(ns, \"make_schedule\")(300, 1e-4, 0.05)\n        mu, s = torch.tensor([1.0, -2.0]), torch.tensor([0.3, 0.8])\n        torch.manual_seed(0)\n        x = need(ns, \"ddpm_sample\")(Oracle(abar, mu, s), betas, abar, 8000)\n        ensure((x.mean(0) - mu).abs().max() < 0.05 and (x.std(0) - s).abs().max() < 0.05,\n               f\"ortalama {x.mean(0).tolist()}, std {x.std(0).tolist()}\")\n    R.test(\"ddpm_sample kâhinle Gauss veriyor (eğitimsiz)\", oracle)\n\n    def train():\n        torch.manual_seed(0)\n        betas, abar = need(ns, \"make_schedule\")()\n        m = need(ns, \"EpsNet\", \"sınıf\")()\n        opt = torch.optim.Adam(m.parameters(), lr=2e-3)\n        for _ in range(3000):\n            l = ns[\"ddpm_loss\"](m, ring(512), abar)\n            opt.zero_grad(); l.backward(); opt.step()\n        x = ns[\"ddpm_sample\"](m, betas, abar, 2000)\n        ang = torch.arange(8) * 2 * math.pi / 8\n        C = torch.stack([2 * ang.cos(), 2 * ang.sin()], 1)\n        dmin, k = torch.cdist(x, C).min(1)\n        good = dmin < 0.2\n        modes = (torch.bincount(k[good], minlength=8) >= 40).sum().item()\n        q = good.float().mean().item()\n        ensure(modes == 8 and q > 0.8, f\"{modes}/8 mod, yakın örnek oranı %{100 * q:.0f}\")\n        return f\"{modes}/8 mod, bir moda 0.2'den yakın örnek oranı %{100 * q:.0f}\"\n    R.test(\"eğitim: 8 Gauss halkası\", train)\n",
 "e": "import math\nimport torch\nfrom torch import nn\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef ring(n):\n    k = torch.randint(0, 8, (n,))\n    ang = k * 2 * math.pi / 8\n    return torch.stack([2 * ang.cos(), 2 * ang.sin()], 1) + 0.05 * torch.randn(n, 2)\n\n\nclass VNet(nn.Module):\n    def __init__(self, h=128, td=32):\n        super().__init__()\n        half = td // 2\n        self.register_buffer(\"f\", torch.exp(-math.log(1000) * torch.arange(half) / half))\n        self.net = nn.Sequential(nn.Linear(2 + td, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))\n\n    def forward(self, x, t):\n        a = (t * 1000)[:, None] * self.f\n        return self.net(torch.cat([x, a.sin(), a.cos()], 1))\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"fm_loss döngüsüz\", lambda: max_loops(src, 0, func=\"fm_loss\"))\n    R.test(\"euler_sample: tek döngü\", lambda: max_loops(src, 1, func=\"euler_sample\"))\n\n    def lf():\n        g = torch.Generator().manual_seed(0)\n        W = torch.randn(3, 3, generator=g)\n        model = lambda x, t: x @ W + t[:, None]\n        x1, z = torch.randn(5, 3, generator=g), torch.randn(5, 3, generator=g); t = torch.rand(5, generator=g)\n        xt = (1 - t[:, None]) * z + t[:, None] * x1\n        close(need(ns, \"fm_loss\")(model, x1, z, t), ((model(xt, t) - (x1 - z)) ** 2).mean())\n    R.test(\"fm_loss formülü\", lf)\n    mu, s = 3.0, 0.5\n    oracle = lambda x, t: mu + (t[:, None] * s ** 2 - (1 - t[:, None])) / ((1 - t[:, None]) ** 2 + t[:, None] ** 2 * s ** 2) * (x - t[:, None] * mu)\n\n    def steps():\n        torch.manual_seed(0)\n        z = torch.randn(20000, 1, dtype=torch.float64)\n        rows = []\n        for n in (1, 2, 5, 20, 100):\n            x = need(ns, \"euler_sample\")(oracle, z, n)\n            rows.append([str(n), f\"{x.mean().item():.3f}\", f\"{x.std().item():.3f}\"])\n        x = ns[\"euler_sample\"](oracle, z, 100)\n        ensure(abs(x.mean().item() - mu) < 0.02 and abs(x.std().item() - s) < 0.02, f\"100 adımda std {x.std().item():.3f}\")\n        x1 = ns[\"euler_sample\"](oracle, z, 1)\n        ensure(x1.std().item() < 1e-6, \"tek adım her örneği ortalamaya götürmeli (kâhinle)\")\n        R.table(\"Kâhin hızla Euler: hedef N(3, 0.5²)\", [\"adım\", \"ortalama\", \"std\"], rows,\n                note=\"Tek adım koşullu ortalamaya gider: varyans 0. Adım arttıkça doğru dağılıma yaklaşır.\")\n    R.test(\"kâhin hızla adım sayısı\", steps)\n\n    def train():\n        torch.manual_seed(0)\n        m = VNet()\n        opt = torch.optim.Adam(m.parameters(), lr=2e-3)\n        for _ in range(3000):\n            x1 = ring(512)\n            l = need(ns, \"fm_loss\")(m, x1, torch.randn_like(x1), torch.rand(512))\n            opt.zero_grad(); l.backward(); opt.step()\n        x = ns[\"euler_sample\"](m, torch.randn(2000, 2), 50)\n        ang = torch.arange(8) * 2 * math.pi / 8\n        C = torch.stack([2 * ang.cos(), 2 * ang.sin()], 1)\n        dmin, k = torch.cdist(x, C).min(1)\n        good = dmin < 0.2\n        modes = (torch.bincount(k[good], minlength=8) >= 40).sum().item()\n        q = good.float().mean().item()\n        ensure(modes == 8 and q > 0.7, f\"{modes}/8 mod, yakın örnek oranı %{100 * q:.0f}\")\n        return f\"50 Euler adımı: {modes}/8 mod, yakın örnek oranı %{100 * q:.0f} (DDPM'de 100 adım)\"\n    R.test(\"eğitim: 8 Gauss halkası\", train)\n",
 "x1": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        betas = np.linspace(1e-4, 0.02, 1000)\n        rows = []\n        for v0 in (0.0, 1.0, 25.0):\n            vs = need(ns, \"var_recursion\")(v0, betas, True)\n            vu = ns[\"var_recursion\"](v0, betas, False)\n            rows.append([v0, f\"{vs[99]:.3f}\", f\"{vs[-1]:.4f}\", f\"{vu[-1]:.2f}\"])\n            close(vs[-1], 1.0, atol=1e-3, what=f\"v₀ = {v0}: ölçekli varyans → 1\")\n        rng = np.random.default_rng(0)\n        x = need(ns, \"forward_chain\")(5.0, betas, rng, 20000)\n        ab = np.cumprod(1 - betas)[-1]\n        close((x.mean(), x.var()), (5 * np.sqrt(ab), 1 - ab), atol=0.03, what=\"zincir = kısa yol: N(√ᾱ x₀, 1 − ᾱ)\")\n        s = need(ns, \"snr\")(np.cumprod(1 - betas))\n        ensure(s[0] > 1e3 and s[-1] < 1e-3, \"SNR yüksekten ~0'a\")\n        R_.table(\"Varyans özyinelemesi, doğrusal β (10⁻⁴ → 0.02), T = 1000\", [\"başlangıç varyansı\", \"t = 100 (ölçekli)\", \"t = 1000 (ölçekli)\", \"t = 1000 (ölçeksiz)\"], rows,\n                 note=\"Ölçeklenmiş süreçte varyans (1 − β)v + β ile 1'e doğru kasılır: başlangıç dağılımı ne olursa olsun x_T ≈ N(0, 1) — örneklemenin başlangıç noktası verinin ölçeğinden bağımsızdır. Ölçeksiz süreçte varyans Σβ kadar artar ve veri ölçeğine bağlı kalır (bu, 'variance exploding' SDE'lerin seçimidir; DDPM 'variance preserving'dir).\")\n    R_.test(\"ileri süreç\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        betas = np.linspace(1e-4, 0.02, 1000)\n        rng = np.random.default_rng(0)\n        rows = []\n        for t_, x0, xt in ((10, 2.0, 1.95), (300, 2.0, 0.5), (900, -1.0, 0.3)):\n            m, v = need(ns, \"posterior\")(x0, xt, t_, betas)\n            mm, mv = need(ns, \"mc_posterior\")(x0, xt, t_, betas, rng, 2000000, 0.01)\n            close((mm, mv), (m, v), atol=0.02 + 0.05 * abs(m), what=f\"t = {t_}: Monte Carlo ≈ kapalı form\")\n            rows.append([t_, f\"{m:.4f}\", f\"{mm:.4f}\", f\"{v:.5f}\", f\"{mv:.5f}\", f\"{betas[t_]:.5f}\"])\n        R_.table(\"q(x_(t−1) | x_t, x₀): kapalı form vs ret örneklemesi\", [\"t\", \"μ̃\", \"MC ortalama\", \"β̃\", \"MC varyans\", \"β_t\"], rows,\n                 note=\"Ağın görevi bu posteriorun ortalamasını x₀ bilinmeden tahmin etmektir: x₀ yerine ε̂'dan kurulan x̂₀ konunca DDPM'in geri adım formülü çıkar. β̃_t ≤ β_t: küçük t'de çok farklı (t = 0'da β̃ = 0), büyük t'de neredeyse eşit — iki varyans seçeneği arasındaki fark yalnız son birkaç adımda önemlidir.\")\n    R_.test(\"geri adımın kapalı formu\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m5_3_diffusion as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**İleri süreç:** $q(x_t\mid x_{t-1})=\mathcal N(\sqrt{\alpha_t}\,x_{t-1},\beta_t I)$, $\alpha_t=1-\beta_t$. Adımlar Gauss olduğu için kapalı form:

$$
x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon,\qquad \bar\alpha_t=\prod_{s\le t}\alpha_s,\ \ \varepsilon\sim\mathcal N(0,I).
$$

**Eğitim:** rastgele $t$ ve $\varepsilon$ seç, $x_t$'yi kur, ağ $\varepsilon_\theta(x_t,t)$ gürültüyü tahmin etsin: $\ \mathcal L=\lVert\varepsilon-\varepsilon_\theta(x_t,t)\rVert^2$.

**DDPM örnekleme** ($t=T,\dots,1$):

$$
x_{t-1}=\frac1{\sqrt{\alpha_t}}\Big(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon_\theta\Big)+\sigma_t z,\qquad \sigma_t^2=\tilde\beta_t=\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\beta_t .
$$

**DDIM (η = 0):** $\hat x_0=(x_t-\sqrt{1-\bar\alpha_t}\,\varepsilon_\theta)/\sqrt{\bar\alpha_t}$, $\ x_{\text{önceki}}=\sqrt{\bar\alpha_{\text{önceki}}}\,\hat x_0+\sqrt{1-\bar\alpha_{\text{önceki}}}\,\varepsilon_\theta$.

**flow matching:** $z\sim\mathcal N(0,I)$'den veriye doğrusal yol $x_t=(1-t)z+tx_1$; ağ hızı öğrenir: $\ \mathcal L=\lVert v_\theta(x_t,t)-(x_1-z)\rVert^2$. Örnekleme: $\dot x=v_\theta(x,t)$'yi $t=0\to1$ Euler ile çöz.

**Kâhin:** veri $\mathcal N(\mu,s^2)$ ise optimal $\varepsilon$-tahmincisi kapalı formda: $\ x_t\sim\mathcal N(\sqrt{\bar\alpha}\mu,\ \bar\alpha s^2+1-\bar\alpha)$ ve

$$
\mathbb E[\varepsilon\mid x_t]=\frac{\sqrt{1-\bar\alpha}\,(x_t-\sqrt{\bar\alpha}\,\mu)}{\bar\alpha s^2+1-\bar\alpha}.
$$

Bununla örnekleyiciyi, eğitim hatasından bağımsız olarak test edebilirsin.

## 1 · Elle

Kodda $t$ 0'dan başlar: `abar[t]` $=\prod_{s=0}^{t}(1-\beta_s)$, ve $t=0$'da $\bar\alpha_{t-1}=1$ sayılır, yani son adımda gürültü eklenmez.

Flow matching sorusunda $x_t=0$ ama hız hedefi her $t$'de aynı, $x_1-z=4$: tek bir $(z,x_1)$ çifti için yol düz bir çizgi.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**$\beta_1=0.1$, $\beta_2=0.2$; $x_0=2$, $\varepsilon=0.5$ (1'den başlayan $t$)**

- `ab2`: $\bar\alpha_2=(1-\beta_1)(1-\beta_2)$
- `x2`: $x_2=\sqrt{\bar\alpha_2}\,x_0+\sqrt{1-\bar\alpha_2}\,\varepsilon$
- `x0h`: $\hat x_0=(x_2-\sqrt{1-\bar\alpha_2}\,\varepsilon)/\sqrt{\bar\alpha_2}$

**Geri adım (doğru $\varepsilon$ ile): $\mu=\frac1{\sqrt{\alpha_2}}\big(x_2-\frac{\beta_2}{\sqrt{1-\bar\alpha_2}}\varepsilon\big)$**

- `mu`: $\mu$
- `var`: $\tilde\beta_2=\frac{1-\bar\alpha_1}{1-\bar\alpha_2}\beta_2$
- `fmv`: Flow matching: $z=-1$, $x_1=3$, $t=0.25$ için $x_t=(1-t)z+tx_1$

In [ ]:
cevaplar = dict(
    ab2=None,
    x2=None,
    x0h=None,
    mu=None,
    var=None,
    fmv=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"ab2":[0.72,0],"x2":[1.961632,0.0005],"x0h":[2,0.0005],"mu":[1.981884,0.0005],"var":[0.071429,0.0005],"fmv":[0,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, skaler $x$; $t$ 0'dan başlar.

- `alpha_bars(betas)` → liste.
- `q_sample(x0, t, eps, abar)`.
- `oracle_eps(xt, t, abar, mu0, s0)` — veri $\mathcal N(\mu_0,s_0^2)$ iken optimal tahmin (yukarıdaki formül).
- `ddpm_step(xt, t, eps_pred, betas, abar, z)` → $x_{t-1}$; `t == 0` iken gürültü yok.
- `sample(eps_fn, betas, n, rng)` — `eps_fn(x, t)`; $x_T\sim\mathcal N(0,1)$ (`rng.gauss(0, 1)`), $t=T-1,\dots,0$.

Kontrol kâhinle 3000 örnek üretip ortalama ve standart sapmanın $\mathcal N(3, 0.5^2)$'ye uyduğunu sınar.

In [ ]:
%%writefile stage_b.py
import math
import random


def alpha_bars(betas):
    ...


def q_sample(x0, t, eps, abar):
    ...


def oracle_eps(xt, t, abar, mu0, s0):
    ...


def ddpm_step(xt, t, eps_pred, betas, abar, z):
    ...


def sample(eps_fn, betas, n, rng):
    ...


T = 100
betas = [1e-4 + (0.1 - 1e-4) * i / (T - 1) for i in range(T)]
ab = alpha_bars(betas)
xs = sample(lambda x, t: oracle_eps(x, t, ab, 3.0, 0.5), betas, 2000, random.Random(0))
m = sum(xs) / len(xs)
print(m, (sum((x - m) ** 2 for x in xs) / len(xs)) ** 0.5)   # ≈ 3, 0.5

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy; tüm örnekler paralel, **yalnız zaman adımları üzerinde tek döngü**.

- `q_sample(x0, t, eps, abar)` — `x0: (n, d)`, `t: (n,)` tamsayı (örnek başına farklı $t$!). Döngü yok.
- `ddpm_sample(eps_fn, betas, n, d, rng)` — `eps_fn(x, t)`, `x: (n, d)`, `t` skaler.
- `ddim_sample(eps_fn, betas, n, d, rng, steps)` — η = 0; zaman adımları `np.linspace(T - 1, 0, steps).round().astype(int)`; son adımdan sonra $\bar\alpha_{\text{önceki}}=1$ (sonuç $\hat x_0$). Rastgelelik yalnız başlangıçta.

Kontrol iki boyutlu Gauss veri için kâhinle iki örnekleyiciyi de sınar ve DDIM'de adım sayısının etkisini tabloya döker. Örnekte $T=300$: $\bar\alpha_T\approx5\cdot10^{-4}$. $\bar\alpha_T$ sıfıra yeterince yakın değilse $x_T\sim\mathcal N(0,I)$ başlangıcı yanlış olur ve hata (özellikle deterministik DDIM'de) doğrudan örneklere taşınır.

In [ ]:
%%writefile stage_c.py
import numpy as np


def q_sample(x0, t, eps, abar):
    ...


def ddpm_sample(eps_fn, betas, n, d, rng):
    ...


def ddim_sample(eps_fn, betas, n, d, rng, steps):
    ...


mu, s = np.array([2.0, -1.0]), np.array([0.5, 1.5])
betas = np.linspace(1e-4, 0.05, 300)
abar = np.cumprod(1 - betas)
oracle = lambda x, t: np.sqrt(1 - abar[t]) * (x - np.sqrt(abar[t]) * mu) / (abar[t] * s ** 2 + 1 - abar[t])
x = ddim_sample(oracle, betas, 4000, 2, np.random.default_rng(0), 50)
print(x.mean(0), x.std(0))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch, 8 Gauss'lu halka üzerinde DDPM.

- `make_schedule(T=100, beta1=1e-4, betaT=0.05)` → `(betas, abar)` (doğrusal).
- `TimeEmbedding(dim)` — sinüzoidal: frekanslar $f_k=\exp(-\log(1000)\,k/(\text{dim}/2))$, çıktı $[\sin(tf),\cos(tf)]$.
- `EpsNet(hidden=128, t_dim=32)` — `forward(x, t)`; `t` uzun tamsayı tensörü `(B,)`. Girdi `cat([x, emb(t)])`, üç gizli katman `SiLU`.
- `ddpm_loss(model, x0, abar, t=None, eps=None)` — verilmezse `t ~ U{0..T-1}`, `eps ~ N(0, I)`; ortalama kare hata.
- `ddpm_sample(model, betas, abar, n)` → `(n, 2)`; `@torch.no_grad()`.

Kontrol önce örnekleyiciyi kâhinle (eğitimsiz) sınar, sonra 3000 adım eğitip halkanın 8 modunun üretildiğini ölçer.

In [ ]:
%%writefile stage_d.py
import math
import torch
from torch import nn


def make_schedule(T=100, beta1=1e-4, betaT=0.05):
    ...


class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        ...

    def forward(self, t):
        ...


class EpsNet(nn.Module):
    def __init__(self, hidden=128, t_dim=32):
        super().__init__()
        ...

    def forward(self, x, t):
        ...


def ddpm_loss(model, x0, abar, t=None, eps=None):
    ...


@torch.no_grad()
def ddpm_sample(model, betas, abar, n):
    ...


def ring(n):
    k = torch.randint(0, 8, (n,))
    ang = k * 2 * math.pi / 8
    return torch.stack([2 * ang.cos(), 2 * ang.sin()], 1) + 0.05 * torch.randn(n, 2)


torch.manual_seed(0)
betas, abar = make_schedule()
model = EpsNet()
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
for step in range(1000):
    loss = ddpm_loss(model, ring(512), abar)
    opt.zero_grad(); loss.backward(); opt.step()
print(loss.item(), ddpm_sample(model, betas, abar, 5))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

flow matching, tümü batch'li:

- `fm_loss(model, x1, z, t)` — `t: (B,)` ∈ [0, 1]; $x_t=(1-t)z+tx_1$; hedef $x_1-z$; `model(x_t, t)` ile ortalama kare hata. Döngü yok.
- `euler_sample(model, z, steps)` — $t_i=i/\text{steps}$, $x\leftarrow x+\frac1{\text{steps}}\,v(x,t_i)$. Tek döngü (adımlar).

Kontrol kâhin hızla adım sayısının etkisini gösterir ve bir ağı senin kaybınla eğitip halkayı 50 Euler adımında üretir.

In [ ]:
%%writefile stage_e.py
import math
import torch
from torch import nn


def fm_loss(model, x1, z, t):
    """t: (B,) ∈ [0, 1]. Döngü yok."""
    ...


@torch.no_grad()
def euler_sample(model, z, steps):
    """Tek döngü: adımlar."""
    ...


mu, s = 3.0, 0.5
oracle = lambda x, t: mu + (t[:, None] * s ** 2 - (1 - t[:, None])) / ((1 - t[:, None]) ** 2 + t[:, None] ** 2 * s ** 2) * (x - t[:, None] * mu)
z = torch.randn(10000, 1)
for n in (1, 5, 50):
    x = euler_sample(oracle, z, n)
    print(n, x.mean().item(), x.std().item())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · İleri süreç: √(1 − β) ölçeklemesi, varyansın 1'e yakınsaması, SNR

**İleri süreçte ölçekleme** (Géron): $x_t=\sqrt{1-\beta_t}\,x_{t-1}+\sqrt{\beta_t}\,\varepsilon$. Ölçekleme olmasaydı ($x_t=x_{t-1}+\sqrt{\beta_t}\varepsilon$) varyans her adımda $\beta_t$ artar ve sınırsız büyür; ölçekleyince $\mathrm{Var}_t=(1-\beta_t)\mathrm{Var}_{t-1}+\beta_t$ → hangi değerden başlarsa başlasın 1'e yakınsar ve $x_T\approx N(0,I)$ olur. Sinyal-gürültü oranı $\mathrm{SNR}_t=\bar\alpha_t/(1-\bar\alpha_t)$.

NumPy.

- `var_recursion(v0, betas, scaled=True)` → her adımdaki varyans (zaman döngüsü serbest).
- `forward_chain(x0, betas, rng, n)` — adım adım (kısa yol olmadan) ileri süreç: `x0` skaler, n örnek; son adımdaki örnekler.
- `snr(abar)`.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def var_recursion(v0, betas, scaled=True):
    ...


def forward_chain(x0, betas, rng, n):
    ...


def snr(abar):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Geri adımın kapalı formu q(x_(t−1) | x_t, x₀): Monte Carlo doğrulama

**Geri adımın kapalı formu.** Bayes ile $q(x_{t-1}\mid x_t,x_0)=N(\tilde\mu_t,\tilde\beta_t)$:
$$\tilde\mu_t=\frac{\sqrt{\bar\alpha_{t-1}}\beta_t}{1-\bar\alpha_t}x_0+\frac{\sqrt{\alpha_t}(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}x_t,\qquad\tilde\beta_t=\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\beta_t.$$
Ho ve ark. ters süreç varyansı için $\sigma_t^2=\beta_t$ ve $\sigma_t^2=\tilde\beta_t$'yi denedi (ikisi de benzer sonuç; biri $x_0$ deterministikken, diğeri $x_0\sim N(0,I)$ iken optimum).

NumPy.

- `posterior(x0, xt, t, betas)` → `(mean, var)` — $t\ge1$ (t = 0 tabanlı indeks, $\bar\alpha_{-1}=1$).
- `mc_posterior(x0, xt, t, betas, rng, n, tol)` — $x_{t-1}\sim q(x_{t-1}|x_0)$, $x_t\sim q(x_t|x_{t-1})$ çiftlerinden $|x_t-\text{xt}|<\text{tol}$ olanları seçip $x_{t-1}$'in ortalama ve varyansı (ret örneklemesi).

In [ ]:
%%writefile extra_x2.py
import numpy as np


def posterior(x0, xt, t, betas):
    ...


def mc_posterior(x0, xt, t, betas, rng, n, tol):
    ...

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

## Makale ödevi: Ho, Jain, Abbeel (2020): DDPM

[Ho, Jain, Abbeel — Denoising Diffusion Probabilistic Models (NeurIPS 2020)](https://arxiv.org/abs/2006.11239)

Küçük bir U-Net (zaman embedding'i, artık bloklar, 16×16'da self-attention) ile MNIST ya da CIFAR-10'da DDPM eğit (T = 1000, doğrusal β, ε-tahmini, basitleştirilmiş kayıp L_simple). Makale Tablo 2'nin ablasyonunu küçük ölçekte yap: ε- vs μ̃-tahmini, sabit σ² = β vs β̃ (x2), L_simple vs ağırlıklı VLB; FID'i (M5.7) raporla. Şekil 6'daki "aşamalı üretim" görselini (x̂₀'ın t'ye göre değişimi) üret.


**Adımlar**

1. EMA'lı ağırlıklarla örnekle (makale: 0.9999).

**Kabul ölçütleri**

- ε-tahmini + L_simple en iyi FID.
- Büyük t'de x̂₀ kaba yapıyı, küçük t'de ayrıntıyı belirler.

## Makale ödevi: Song, Meng, Ermon (2021): DDIM

[Song, Meng, Ermon — Denoising Diffusion Implicit Models (ICLR 2021)](https://arxiv.org/abs/2010.02502)

Yukarıdaki eğitilmiş DDPM'i yeniden eğitmeden DDIM ile örnekle: adım sayısı S ∈ {10, 20, 50, 100, 1000} ve η ∈ {0, 0.5, 1} için FID tablosu (makale Tablo 1). η = 0'da aynı x_T'den farklı S ile üretilen görüntülerin tutarlılığını ve gizli uzayda interpolasyonu (Şekil 6, slerp — M5.1 x3) göster.


**Adımlar**

1. Alt dizi seçimi: doğrusal ve karesel aralıklı τ.

**Kabul ölçütleri**

- η = 0, az adımda DDPM'den çok daha iyi FID.
- Aynı x_T farklı S'lerde benzer üst düzey yapı üretir.

## Kaynaklar

- [Ho, Jain, Abbeel — Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239) — arXiv
- [Song, Meng, Ermon — Denoising Diffusion Implicit Models](https://arxiv.org/abs/2010.02502) — arXiv
- [Lipman ve ark. — Flow Matching for Generative Modeling](https://arxiv.org/abs/2210.02747) — arXiv
- [Liu, Gong, Liu — Rectified Flow](https://arxiv.org/abs/2209.03003) — arXiv
- [fast.ai Part 2 — From Deep Learning Foundations to Stable Diffusion](https://course.fast.ai/Lessons/part2.html) — course22p2 kodu Apache-2.0
- [Prince — Understanding Deep Learning, Bölüm 18 (Diffusion)](https://udlbook.github.io/udlbook/) — kitap ©; notebook'lar MIT
- [MIT 6.S184 — Generative AI with Stochastic Differential Equations (flow matching notları)](https://diffusion.csail.mit.edu/) — doğrulanmadı